In [ ]:
from pathlib import Path

from IPython.display import display
from pydantic import model_validator

from src.common.training import TrainingConfig
from src.config.base import BaseConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.base import ModelConfig
from src.model.mlp import StackedResidualMLPConfig


In [ ]:
class GaussianDriftingProtocolConfig(BaseConfig):
    """
    The drifting field is fixed to use Gaussian kernel smoothing.
    Relative to chart transport, the protocol surface is much smaller:
    1. kernel_width determines the KDE smoothing scale.
    2. transport_step_size determines the stop-grad transport target size.
    """

    kernel_width: float
    transport_step_size: float

    @model_validator(mode="after")
    def _validate_config(self) -> "GaussianDriftingProtocolConfig":
        if self.kernel_width <= 0.0:
            raise ValueError("kernel_width must be positive")
        if self.transport_step_size <= 0.0:
            raise ValueError("transport_step_size must be positive")
        return self


class DriftingExperimentConfig(BaseConfig):
    data_config: MultimodalGaussianDataConfig
    decoder_config: ModelConfig
    protocol_config: GaussianDriftingProtocolConfig


def get_canonical_drifting_config(
    *,
    data_config: MultimodalGaussianDataConfig,
) -> DriftingExperimentConfig:
    hidden_dimension = 256

    decoder_config = StackedResidualMLPConfig.initialize(
        layer_dims=[
            data_config.data_numel(),
            hidden_dimension,
            hidden_dimension,
            data_config.data_numel(),
        ],
    )
    protocol_config = GaussianDriftingProtocolConfig(
        kernel_width=0.75,
        transport_step_size=1.0,
    )
    return DriftingExperimentConfig(
        data_config=data_config,
        decoder_config=decoder_config,
        protocol_config=protocol_config,
    )


In [ ]:
num_modes = 4
mode_std = 0.1
offset = 1.0
ambient_dimension = 8

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

display(data_config.visualize())

drifting_config = get_canonical_drifting_config(
    data_config=data_config,
)
training_config = TrainingConfig(
    train_batch_size=2048,
    eval_batch_size=1024,
    eval_every_n_training_steps=3000,
    folder=Path("runs/drifting-multimodal-gaussian"),
    raise_on_existing_folder=False,
)

display(drifting_config.visualize())
display(training_config.visualize())


## Field Evaluation

For each model sample `x`, evaluate the Gaussian-kernel attraction to data and repulsion from model samples with shared `kernel_width`, then form the drifting target `x + transport_step_size * V(x)`.


## Protocol Notes

Unlike chart transport, plain drifting has no encoder, no manifold constraint, and no noise critic. The protocol-specific config therefore stays minimal here; if we later want to expose forward-KL density-ratio reweighting as a separate variant, that should be added as a distinct config leaf rather than folded into the two core hyperparameters above.
